# 歡迎來到 Day 2 Lab！


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">開始之前——</h2>
            <span style="color:#f71;">我想花一秒鐘介紹本課程實用資源頁，包含所有投影片連結。<br/>
            <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">https://edwarddonner.com/2024/11/13/llm-engineering-resources/</a><br/>
            請加入書籤，我會持續新增實用連結。
            </span>
        </td>
    </tr>
</table>

## 首先——聊聊 Chat Completions API

1. 呼叫 LLM 最簡單的方式
2. 稱為 Chat Completions，意思是：「這是一段對話，請預測接下來該說什麼」
3. Chat Completions API 由 OpenAI 發明，但太受歡迎，大家都用它！

### 我們先再次呼叫 OpenAI——非 OpenAI 使用者別擔心，很快就輪到你！


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


## 你知道什麼是 Endpoint 嗎？

若不知道，請複習 guides 資料夾中的 Technical Foundations 指南

這裡有一個你可能感興趣的 endpoint……

In [ ]:
import requests

headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}

payload = {
    "model": "gpt-5-nano",
    "messages": [
        {"role": "user", "content": "Tell me a fun fact"}]
}

payload

In [ ]:
response = requests.post(
    "https://api.openai.com/v1/chat/completions",
    headers=headers,
    json=payload
)

response.json()

In [ ]:
response.json()["choices"][0]["message"]["content"]

# openai 套件是什麼？

它稱為 Python Client Library（Python 客戶端函式庫）。

它只是對向此 HTTP endpoint 發送請求的輕量包裝。

讓你用乾淨的 Python 程式碼工作，而不必處理繁瑣的 JSON 物件。

僅此而已。它是開源且輕量的。有人以為它包含 OpenAI 模型程式碼——並沒有！


In [ ]:
# 建立 OpenAI 客戶端

from openai import OpenAI
openai = OpenAI()

response = openai.chat.completions.create(model="gpt-5-nano", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content



## 然後發生了這件很棒的事：

OpenAI 的 Chat Completions API 太受歡迎，其他模型供應商也建立了相同的 endpoint。

它們稱為「OpenAI Compatible Endpoints」（OpenAI 相容端點）。

例如 Google 在這裡提供：https://generativelanguage.googleapis.com/v1beta/openai/

OpenAI 也很友善：他們說，你可以繼續用我們為 GPT 做的同一個客戶端函式庫，只需指定不同的 endpoint URL 與金鑰，即可使用其他供應商。

因此你可以：

```python
gemini = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/", api_key="AIz....")
gemini.chat.completions.create(...)
```

請注意——即使程式碼中有 OpenAI，我們只是用這個輕量 Python 客戶端呼叫 endpoint——這裡沒有 OpenAI 模型參與。

若感到困惑，請複習 Guides 資料夾的 Guide 9！

現在來試試！

## 以下為選修——若你想試用 Google Gemini，請造訪：

https://aistudio.google.com/

並在此設定 API 金鑰

https://aistudio.google.com/api-keys

然後將金鑰加入 `.env` 檔，變更後務必儲存 .env：

`GOOGLE_API_KEY=AIz...`


In [ ]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

load_dotenv(override=True)

google_api_key = os.getenv("GOOGLE_API_KEY")

if not google_api_key:
    print("No API key was found - please be sure to add your key to the .env file, and save the file! Or you can skip the next 2 cells if you don't want to use Gemini")
elif not google_api_key.startswith("AIz"):
    print("An API key was found, but it doesn't start AIz")
else:
    print("API key found and looks good so far!")



In [ ]:
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)

response = gemini.chat.completions.create(model="gemini-2.5-flash-lite", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

## Ollama 也提供 OpenAI 相容 endpoint

……而且就在你的本機！

若下一個儲存格沒有印出「Ollama is running」，請開啟終端機執行 `ollama serve`

In [ ]:
requests.get("http://localhost:11434").content

### 從 Meta 下載 llama3.2

若電腦較小，可改為 llama3.2:1b。

不要用 llama3.3 或 llama4！對你的電腦來說太大了……

In [ ]:
!ollama pull llama3.2

In [ ]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')


In [ ]:
# 取得一則趣味小知識

response = ollama.chat.completions.create(model="llama3.2", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

In [ ]:
# 現在試試 deepseek-r1:1.5b——這是 DeepSeek「蒸餾」進阿里巴巴雲的 Qwen

!ollama pull deepseek-r1:1.5b

In [ ]:
response = ollama.chat.completions.create(model="deepseek-r1:1.5b", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

# 作業練習

將 Day 1 的網頁摘要專案升級為透過 Ollama 在本機執行開源模型，而非 OpenAI

若你偏好不使用付費 API，此技巧可套用於後續所有專案。

**優點：**
1. 無 API 費用——開源
2. 資料不離開你的機器

**缺點：**
1. 能力明顯低於前沿模型

## Ollama 安裝複習

造訪 [ollama.com](https://ollama.com) 並安裝即可！

完成後，ollama 伺服器應已在本地執行。  
造訪：  
[http://localhost:11434/](http://localhost:11434/)

應看到 `Ollama is running` 訊息。  

若沒有，開啟新終端機（Mac）或 Powershell（Windows）輸入 `ollama serve`  
再在另一個終端機（Mac）或 Powershell（Windows）輸入 `ollama pull llama3.2`  
然後再試 [http://localhost:11434/](http://localhost:11434/)。

若 Ollama 在你的機器上很慢，可試 `llama3.2:1b`。在終端機或 Powershell 執行 `ollama pull llama3.2:1b`，並將程式碼從 `MODEL = "llama3.2"` 改為 `MODEL = "llama3.2:1b"`